# 🔍 MOVE Research Agent

**Unified pipeline — competitors are discovered automatically:**
```
START
  ↓
Market Insight Scanner     ← scrape company, extract context
  ↓
Competitor Discovery       ← auto-find + profile competitors
  ↓
Competitor Tracker         ← compare, pricing analysis
  ↓
Trend Discovery Engine     ← trends, reviews, pain points, gaps
  ↓
Research Synthesis         ← ResearchReport + insights
  ↓
END
```

In [ ]:
# ── Must run first ──────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()   # allows asyncio in Jupyter

import os, json
from dotenv import load_dotenv
load_dotenv()

COMPANY_URL  = "https://beamdata.ai/"   # ← change this
COMPANY_NAME = "BeamData"               # ← change this

print(f"✅ Ready")
print(f"   OpenAI:    {'set' if os.getenv('OPENAI_API_KEY') else '❌ MISSING'}")
print(f"   Firecrawl: {'set' if os.getenv('FIRECRAWL_API_KEY') else 'not set (crawl4ai fallback)'}")
print(f"   LangSmith: {'set' if os.getenv('LANGCHAIN_API_KEY') else 'not set'}")


## ⚡ Option A — Run Full Pipeline (One Shot)
Competitors are found automatically. No need to supply URLs.

In [ ]:
from backend.agents.research.research_agent import run_research

result = run_research(COMPANY_URL, COMPANY_NAME)

print("✅ Pipeline complete!")
print(f"   Competitors profiled: {len(result['competitors'])}")
print(f"   Report sections: {list(result['report'].keys())}")
print(f"\n--- EXECUTIVE SUMMARY ---")
print(result['insights'].get('executive_summary', 'N/A'))

## 🔬 Option B — Step by Step
Run each node individually to inspect outputs at each stage.

In [ ]:
# Step 1: Market Insight Scanner
from backend.agents.research.scanners.market_insight_scanner import run as market_scan

state = {"company_url": COMPANY_URL, "company_name": COMPANY_NAME}
state = market_scan(state)

ctx = state["company_context"]
print(f"Company:     {ctx.get('company_name')}")
print(f"Industry:    {ctx.get('industry')}")
print(f"Products:    {ctx.get('products')}")
print(f"Market:      {ctx.get('target_market')}")
print(f"Pricing:     {ctx.get('pricing_model')}")
print(f"Positioning: {ctx.get('unique_positioning')}")

In [ ]:
# Step 2: Competitor Discovery (auto-find)
from backend.agents.research.research_agent import competitor_discovery_node

state = competitor_discovery_node(state)

print(f"Competitors discovered: {len(state['competitors_discovered'])}")
print(f"Competitors profiled:   {len(state['competitors_profiled'])}")
print()
for c in state["competitors_profiled"]:
    print(f"  • {c['name']} — {c['website']}")
    print(f"    Positioning: {c.get('positioning', 'N/A')}")
    print(f"    Pricing:     {c.get('pricing_model', 'N/A')}")
    print(f"    Features:    {c.get('key_features', [])[:3]}")
    print()

In [ ]:
# Step 3: Competitor Tracker (compare + pricing)
from backend.agents.research.scanners.competitor_tracker import run as comp_track

state = comp_track(state)

comp = state["competitor_comparison"]
print("Our Advantages:")
for a in comp.get("our_advantages", []):
    print(f"  ✅ {a}")

print("\nFeature Gaps:")
for g in comp.get("feature_gaps", []):
    print(f"  ⚠️  {g}")

print("\nMarket Gaps:")
for g in comp.get("market_gaps", []):
    print(f"  💡 {g}")

pricing = state["pricing_analysis"]
print(f"\nMarket Price Range: {pricing.get('market_price_range', {})}")
print(f"Pricing Position:   {comp.get('pricing_position', 'N/A')}")

In [ ]:
# Step 4: Trend Discovery Engine
from backend.agents.research.scanners.trend_discovery_engine import run as trend_run

state = trend_run(state)

print("TOP TRENDS:")
for t in state["trends_clustered"].get("top_priority_trends", [])[:5]:
    print(f"  📈 {t}")

print("\nCUSTOMER PAIN POINTS:")
for p in state["pain_points"].get("top_3_critical", []):
    print(f"  😣 {p}")

print("\nTOP OPPORTUNITIES:")
for o in state["scored_opportunities"].get("scored_opportunities", [])[:3]:
    print(f"  🚀 [{o.get('composite_score')}/10] {o.get('opportunity')}")

In [ ]:
# Step 5: Research Synthesis
from backend.agents.research.research_agent import research_synthesis

state = research_synthesis(state)
report  = state["report"]
insights = state["insights"]

print("=" * 70)
print("EXECUTIVE SUMMARY")
print("=" * 70)
print(insights.get("executive_summary", "N/A"))

print("\nSTRATEGIC RECOMMENDATIONS")
for r in report.get("strategic_recommendations", []):
    print(f"  #{r.get('priority')} {r.get('recommendation')}")
    print(f"     → {r.get('timeline')}")

print(f"\n🏆 Best Opportunity: {insights.get('best_opportunity')}")
print(f"⚠️  Biggest Risk:     {insights.get('biggest_risk')}")

## 🗂️ Save & Query


In [ ]:
with open("research_report.json", "w") as f:
    json.dump({"report": report, "insights": insights}, f, indent=2)
print("✅ Saved to research_report.json")

In [ ]:
# Query all stored research
from backend.rag.vector_store import VectorStore
from backend.rag.embedder import embed_text

QUESTIONS = {
    "Company":    ["What does this company do?", "What are its products?", "What is its pricing model?", "What differentiates it?"],
    "Competitors":["Who are the competitors?", "How do competitor prices compare?", "What features do competitors offer?"],
    "Customers":  ["What complaints appear most often?", "What features are requested most?"],
    "Market":     ["What trends are emerging?", "What market gaps exist?", "What opportunities exist?"],
}

vs_kb       = VectorStore(collection_name="knowledge_base")
vs_research = VectorStore(collection_name="research")

for section, questions in QUESTIONS.items():
    print(f"\n{'='*60}")
    print(f"  {section} Questions")
    print(f"{'='*60}")
    vs = vs_kb if section == "Company" else vs_research
    for q in questions:
        vec = embed_text(q)
        top = vs.query(vec, n_results=1)
        if top:
            print(f"\nQ: {q}")
            print(f"   {top[0]['content'].strip()[:300]}")